# 2.2a Train and tune the fraud model

**Story:** The features are ready (notebook 1.2a). Now you train SafeDrive's claim-fraud model on
Amazon SageMaker AI, tune it, make training faster and cheaper, and keep it from overfitting.

**Task 2.2:** Train, fine-tune, and customize models for ML and AI solutions (traditional ML part).

| Skill | Topic | Status |
|---|---|---|
| 2.2.1 | SageMaker built-in algorithms and common ML libraries | ✅ |
| 2.2.2 | Script mode with a supported framework | ✅ |
| 2.2.7 | Basic hyperparameters (epochs, steps, batch size, learning rate) | ✅ |
| 2.2.5 | Prevent overfitting and underfitting (forgetting is in 2.2b) | ✅ |
| 2.2.3 | Hyperparameter optimization (automatic model tuning) | ✅ |
| 2.2.4 | Reduce training time (early stopping, distributed, Spot) | ✅ |
| 2.1.6 | Tradeoff: model performance vs training time vs cost | ✅ |
| 2.2.6 | Combine models (ensemble, stacking, cascade) | ✅ |

💰 About 8 short training jobs on `ml.m5.large` (a few cents each). Total run time about 30 minutes.

## Key points for the exam

- **Built-in algorithm** = AWS container, you give data + hyperparameters. **Script mode** = AWS framework
  container (scikit-learn, XGBoost, PyTorch, TensorFlow, Hugging Face) + your training script. **BYOC** = your own container.
- Built-in XGBoost wants **CSV with the label in the first column and no header** (or libsvm, Parquet, RecordIO-protobuf).
- Data arrives in **channels** (`train`, `validation`). In script mode they are folders named by `SM_CHANNEL_<NAME>`; save the model to `SM_MODEL_DIR`.
- **Automatic model tuning (AMT)** strategies: Bayesian (default, learns from earlier jobs), Random (fully parallel), Grid, Hyperband (stops weak jobs early, good for deep learning).
- **Warm start** reuses results of earlier tuning jobs: `IdenticalDataAndAlgorithm` or `TransferLearning`.
- **Managed Spot training** saves up to 90%. Set `max_wait` ≥ `max_run` and use **checkpoints** so an interrupted job can resume.
- **Overfitting**: train score high, validation score low → more regularization, simpler model, more data, early stopping, dropout.
  **Underfitting**: both low → bigger model, more features, train longer, less regularization.
- **Learning rate** too high → loss jumps or diverges. Too low → very slow training.

| Symptom | Fix |
|---|---|
| Training takes too long on big data | Distributed training (`instance_count` > 1), Pipe / FastFile input mode, GPU |
| Paying too much for training | Managed Spot + checkpoints, early stopping, smaller instance |
| Many repeated short jobs start slowly | SageMaker managed **warm pools** (`keep_alive_period_in_seconds`) |

## Setup

In [ ]:
import sys; sys.path.append("..")
import os, time, warnings
import numpy as np, pandas as pd
from common import utils as U, catchup as C, features as F
warnings.filterwarnings("ignore")
print(U.REGION, U.bucket())

Catch-up: build the train / validation / test CSV files if notebook 1.2a has not been run.

In [ ]:
state = C.ensure_training_data()
print(state["train_s3"], state["validation_s3"], sep="\n")

Create the SageMaker session and client, and load the training and validation sets into pandas for the
quick local experiments. Look at the fraud rate: the classes are imbalanced.

In [ ]:
import sagemaker
sm_session = sagemaker.Session(boto_session=U.session, default_bucket=U.bucket())
sm = U.client("sagemaker")
cols = [F.LABEL] + F.FEATURES
train = U.read_csv(f"{C.PROCESSED}/train/train.csv", header=None, names=cols)
val = U.read_csv(f"{C.PROCESSED}/validation/validation.csv", header=None, names=cols)
Xtr, ytr, Xva, yva = train[F.FEATURES], train[F.LABEL], val[F.FEATURES], val[F.LABEL]
print(train.shape, val.shape, "fraud rate:", round(ytr.mean(), 4))

## 2.2.1 Built-in algorithm: XGBoost training job

- You pick the algorithm **image**, an instance type, input channels and hyperparameters. No training code.
- Output is `model.tar.gz` in S3. The job's instances are deleted when it ends: you pay per second of training.
- Common ML libraries (scikit-learn, XGBoost, PyTorch, TensorFlow) run in **script mode**: see 2.2.2.

Look up the container image of the built-in XGBoost algorithm for this Region.

In [ ]:
from sagemaker.estimator import Estimator
from sagemaker.inputs import TrainingInput
xgb_image = sagemaker.image_uris.retrieve("xgboost", U.REGION, version="1.7-1")
print(xgb_image)

A small helper that builds an XGBoost `Estimator`. We reuse it for every job in this notebook, changing
only what the lesson is about. `scale_pos_weight` gives the rare fraud class more weight.

In [ ]:
BASE_HP = dict(objective="binary:logistic", eval_metric="auc", num_round=150,
               max_depth=5, eta=0.1, subsample=0.8, scale_pos_weight=10)

def make_xgb(base_job_name, hyperparameters=None, **kwargs):
    est = Estimator(image_uri=xgb_image, role=U.role_arn(), instance_type="ml.m5.large",
                    instance_count=kwargs.pop("instance_count", 1),
                    output_path=U.s3_uri(f"{U.PREFIX}/models"), sagemaker_session=sm_session,
                    base_job_name=base_job_name, tags=U.TAGS, **kwargs)
    est.set_hyperparameters(**{**BASE_HP, **(hyperparameters or {})})
    return est

est = make_xgb("safedrive-fraud-xgb")
est.hyperparameters()

The two input channels. Built-in XGBoost reads CSV with the label first and no header.

In [ ]:
inputs = {"train": TrainingInput(state["train_s3"], content_type="text/csv"),
          "validation": TrainingInput(state["validation_s3"], content_type="text/csv")}
{k: v.config["DataSource"]["S3DataSource"]["S3Uri"] for k, v in inputs.items()}

Train. If a baseline model already exists (from a catch-up run), reuse it instead of training again.
The job takes about 4 minutes; most of that is instance start-up.

In [ ]:
if U.get_state("fraud_model_data"):
    print("reusing", U.get_state("fraud_training_job"))
else:
    est.fit(inputs, logs=False)
    U.save_state(fraud_model_data=est.model_data, xgb_image=xgb_image,
                 fraud_training_job=est.latest_training_job.name)
BASE_JOB = U.get_state("fraud_training_job")
print("baseline job:", BASE_JOB, "\nartifact:", U.get_state("fraud_model_data"))

Inspect the finished job. Look at the final metrics, and at **training time vs billable time**.

In [ ]:
d = sm.describe_training_job(TrainingJobName=BASE_JOB)
print(d["TrainingJobStatus"], "| instances:", d["ResourceConfig"]["InstanceCount"],
      "| train s:", d["TrainingTimeInSeconds"], "| billable s:", d["BillableTimeInSeconds"])
pd.DataFrame(d["FinalMetricDataList"])[["MetricName", "Value"]]

> 🖥️ **Console check:** *Amazon SageMaker AI → Training → Training jobs → the baseline job (name printed above)*.
> Look at the input data channels, the hyperparameters, the output artifact path, the training time and billable time, and the metric graphs in the Monitor section. "View logs" opens the job's CloudWatch log stream.

## 2.2.2 Script mode with scikit-learn

- You write a normal training script; SageMaker runs it inside its scikit-learn container.
- Hyperparameters arrive as command-line arguments. Data folders and the model folder arrive as
  environment variables (`SM_CHANNEL_TRAIN`, `SM_MODEL_DIR`).
- Extra libraries: add a `requirements.txt` in `source_dir`. Need full control: bring your own container.
- Print metrics to the log and give a **regex** in `metric_definitions` so SageMaker (and AMT) can read them.

Write the training script: a scaled logistic regression. `model_fn` tells the container how to load
the model later for inference.

In [ ]:
os.makedirs("scripts", exist_ok=True)

In [ ]:
%%writefile scripts/train_sklearn.py
import argparse, glob, os
import joblib
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


def load(folder):
    """Read every CSV in a channel folder: label is the first column, no header."""
    df = pd.concat(pd.read_csv(f, header=None) for f in glob.glob(os.path.join(folder, "*.csv")))
    return df.iloc[:, 1:].values, df.iloc[:, 0].values


def model_fn(model_dir):
    return joblib.load(os.path.join(model_dir, "model.joblib"))


if __name__ == "__main__":
    p = argparse.ArgumentParser()
    p.add_argument("--C", type=float, default=1.0)              # inverse regularization strength
    p.add_argument("--max-iter", type=int, default=500)
    p.add_argument("--model-dir", default=os.environ.get("SM_MODEL_DIR", "model"))
    p.add_argument("--train", default=os.environ.get("SM_CHANNEL_TRAIN"))
    p.add_argument("--validation", default=os.environ.get("SM_CHANNEL_VALIDATION"))
    args = p.parse_args()

    X, y = load(args.train)
    Xv, yv = load(args.validation)
    model = make_pipeline(StandardScaler(),
                          LogisticRegression(C=args.C, max_iter=args.max_iter, class_weight="balanced"))
    model.fit(X, y)
    print(f"validation-auc: {roc_auc_score(yv, model.predict_proba(Xv)[:, 1]):.4f}")
    os.makedirs(args.model_dir, exist_ok=True)
    joblib.dump(model, os.path.join(args.model_dir, "model.joblib"))

Test the script locally first (free and fast). Put small copies of the data in local folders that look
like SageMaker channels, then run the script. Look for the `validation-auc` line.

In [ ]:
for name, part in [("train", train), ("validation", val)]:
    os.makedirs(f"local_data/{name}", exist_ok=True)
    part.to_csv(f"local_data/{name}/{name}.csv", header=False, index=False)
!python scripts/train_sklearn.py --train local_data/train --validation local_data/validation --model-dir local_data/model

Now define the same run as a SageMaker job with the `SKLearn` estimator. The regex turns the printed
line into a metric called `validation:auc`.

In [ ]:
from sagemaker.sklearn import SKLearn
sk = SKLearn(entry_point="train_sklearn.py", source_dir="scripts", framework_version="1.2-1",
             role=U.role_arn(), instance_type="ml.m5.large", instance_count=1,
             hyperparameters={"C": 0.5, "max-iter": 500},
             metric_definitions=[{"Name": "validation:auc", "Regex": "validation-auc: ([0-9\\.]+)"}],
             output_path=U.s3_uri(f"{U.PREFIX}/models"), sagemaker_session=sm_session,
             base_job_name="safedrive-fraud-sklearn", tags=U.TAGS)
print(sk.training_image_uri())

Start the job (about 4 minutes) and wait.

In [ ]:
sk.fit(inputs, logs=False)
SK_JOB = sk.latest_training_job.name
print(SK_JOB, sk.model_data)

Read the metric SageMaker pulled from the log with the regex.

In [ ]:
d = sm.describe_training_job(TrainingJobName=SK_JOB)
pd.DataFrame(d["FinalMetricDataList"])[["MetricName", "Value"]]

## 2.2.7 Basic hyperparameters: rounds/epochs, learning rate, batch size

- **Epoch** = one pass over all training rows. **Batch size** = rows used for one weight update.
  **Step** (iteration) = one update. `steps = epochs × rows / batch size`.
- **Learning rate** = size of each update. In XGBoost it is `eta`, and `num_round` (number of trees) plays the role of epochs.
- Small learning rate needs more rounds/epochs. Large learning rate learns fast but can overshoot.
- Small batch = noisy but frequent updates (often generalizes better, slower per epoch). Large batch =
  fast per epoch and uses hardware well, but may need a higher learning rate or more epochs.

Quick local experiment with XGBoost: validation AUC for three learning rates and two numbers of rounds.
Look at `eta=0.01` with 20 rounds (too few small steps: underfit) and at how the best pair combines the two.

In [ ]:
import xgboost as xgb
from sklearn.metrics import roc_auc_score, average_precision_score

def fit_xgb(**params):
    return xgb.XGBClassifier(eval_metric="auc", **params).fit(
        Xtr, ytr, eval_set=[(Xtr, ytr), (Xva, yva)], verbose=False)

rows = [{"eta": eta, "num_round": n,
         "val_auc": fit_xgb(learning_rate=eta, n_estimators=n, max_depth=5).evals_result()["validation_1"]["auc"][-1]}
        for eta in (0.01, 0.1, 0.5) for n in (20, 150)]
pd.DataFrame(rows).pivot(index="eta", columns="num_round", values="val_auc").round(4)

Epochs and batch size on a tiny neural network (scikit-learn `MLPClassifier`). Look at the number of
steps, the time and the AUC: with a big batch you get far fewer updates per epoch, so you need more epochs.

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

def run_mlp(epochs, batch):
    start = time.time()
    net = make_pipeline(StandardScaler(), MLPClassifier((16,), max_iter=epochs, batch_size=batch, random_state=0))
    net.fit(Xtr, ytr)
    return {"epochs": epochs, "batch_size": batch, "steps": epochs * int(np.ceil(len(Xtr) / batch)),
            "seconds": round(time.time() - start, 1),
            "val_auc": round(roc_auc_score(yva, net.predict_proba(Xva)[:, 1]), 4)}

pd.DataFrame([run_mlp(e, b) for e, b in [(1, 32), (10, 32), (10, 1024), (40, 1024)]])

## 2.2.5 Prevent overfitting and underfitting

- Compare the **training** score with the **validation** score.
- Big gap (train much better) = **overfitting**. Both poor = **underfitting**.
- XGBoost regularization knobs: `max_depth` ↓, `min_child_weight` ↑, `gamma` ↑, `lambda` (L2) ↑, `alpha` (L1) ↑,
  `subsample` / `colsample_bytree` < 1, `eta` ↓, plus **early stopping**.
- Neural networks: dropout, weight decay (L2), early stopping, data augmentation, more data.
- **Catastrophic forgetting** (a fine-tuned model loses what it knew before) is covered in notebook 2.2b.

Train three models locally: too simple, reasonable, too complex. Compare train and validation AUC.

In [ ]:
configs = {
    "underfit (3 stumps)": dict(n_estimators=3, max_depth=1, learning_rate=0.1),
    "balanced": dict(n_estimators=100, max_depth=3, learning_rate=0.1),
    "overfit (deep trees)": dict(n_estimators=200, max_depth=12, learning_rate=0.3),
}
fitted = {name: fit_xgb(**p) for name, p in configs.items()}

def scores(model):
    h = model.evals_result()
    return {"train_auc": h["validation_0"]["auc"][-1], "val_auc": h["validation_1"]["auc"][-1]}

pd.DataFrame({name: scores(m) for name, m in fitted.items()}).T.round(4)

Plot the learning curves of the overfit model. Training AUC climbs to 1.0 while validation AUC peaks
early and then falls: the classic overfitting picture.

In [ ]:
import matplotlib.pyplot as plt
h = fitted["overfit (deep trees)"].evals_result()
plt.figure(figsize=(6, 3))
plt.plot(h["validation_0"]["auc"], label="train")
plt.plot(h["validation_1"]["auc"], label="validation")
plt.xlabel("boosting round"); plt.ylabel("AUC"); plt.legend(); plt.title("Overfitting: the curves split")
plt.show()

**📘 Concept: Regularization, and which knob does what**

Regularization means limiting how closely a model can fit the training data, so it learns the general pattern and not the noise. You trade a slightly worse training score for a better validation score.

| Technique | What it does | Where you see it |
|---|---|---|
| **L2** (ridge, `lambda`, weight decay) | Penalizes large weights; shrinks all of them smoothly | XGBoost `lambda`, Linear Learner `wd`, neural networks |
| **L1** (lasso, `alpha`) | Penalizes absolute weight size; pushes useless weights to exactly zero, so it also **selects features** | XGBoost `alpha`, Linear Learner `l1` |
| **Simpler model** | Fewer or shallower trees, fewer layers | `max_depth`, `min_child_weight`, `gamma` |
| **Randomness** | Each tree or step sees only part of the rows or columns | `subsample`, `colsample_bytree`; **dropout** in neural networks |
| **Early stopping** | Stop when the validation metric stops improving | `early_stopping_rounds`, needs a validation channel |
| **More or augmented data** | Makes noise harder to memorize | Any model |

- Too much regularization causes the opposite problem: underfitting (both scores low).
- Exam cues: "training accuracy high, validation accuracy low" → add regularization, dropout, early stopping or data. "Many irrelevant features, want a sparse model" → L1. "Both training and validation error are high" → reduce regularization or use a more complex model.

Fix it with regularization and early stopping, keeping the same deep trees. Look at how the gap
shrinks, and at the round where early stopping ended training.

In [ ]:
regular = fit_xgb(n_estimators=200, max_depth=12, learning_rate=0.1, min_child_weight=10, gamma=2,
                  reg_lambda=10, subsample=0.7, colsample_bytree=0.7, early_stopping_rounds=10)
print("stopped at round", regular.best_iteration, "of 200")
pd.DataFrame({"overfit": scores(fitted["overfit (deep trees)"]), "regularized": scores(regular)}).T.round(4)

## 2.2.3 Hyperparameter optimization with automatic model tuning (AMT)

- You give **ranges**, an **objective metric** and a budget (`max_jobs`, `max_parallel_jobs`). AMT runs training jobs and picks the best.
- **Bayesian** uses earlier results to choose the next values, so keep parallel jobs low. **Random** / **Grid** can run fully in parallel. **Hyperband** stops weak jobs early.
- Use a **log scale** for values that span orders of magnitude (learning rate, regularization).
- `early_stopping_type="Auto"` stops jobs that are clearly not going to win (saves money).

**📘 Concept: Tuning strategies in automatic model tuning**

| Strategy | How it picks the next values | Choose when |
|---|---|---|
| **Grid** | Tries every combination of the listed values (categorical values only) | Few hyperparameters with a few values each; you need full, repeatable coverage |
| **Random** | Random points in the ranges; jobs are independent | You want maximum parallelism, or a quick broad exploration |
| **Bayesian** (default) | Builds a model of "values → metric" from finished jobs and picks promising values next | Limited budget of jobs; each job is expensive |
| **Hyperband** | Starts many configurations, gives them little resource, stops the weak ones early and moves resource to the strong ones | Iterative training (deep learning, many epochs) that reports metrics while it runs |

- Bayesian search learns from finished jobs. With very high parallelism most jobs start before any result exists, so it behaves more like random search. Fewer parallel jobs = better search, but longer wall-clock time.
- Hyperband has its own early stopping built in; the separate `early_stopping_type` setting is for the other strategies.
- **Warm start** (later in this section) seeds a new tuning job with the results of earlier ones.
- Exam cues: "fewest training jobs to find a good model" → Bayesian. "Run all jobs at the same time" → Random. "Stop under-performing deep learning jobs early" → Hyperband.

Define the search space and the tuner: 4 jobs, 2 at a time, Bayesian strategy.

In [ ]:
from sagemaker.tuner import HyperparameterTuner, ContinuousParameter, IntegerParameter
ranges = {"eta": ContinuousParameter(0.01, 0.3, scaling_type="Logarithmic"),
          "max_depth": IntegerParameter(2, 8),
          "min_child_weight": ContinuousParameter(1, 10),
          "subsample": ContinuousParameter(0.5, 1.0)}
tuner = HyperparameterTuner(
    estimator=make_xgb("safedrive-fraud-hpo"), objective_metric_name="validation:auc",
    objective_type="Maximize", hyperparameter_ranges=ranges, strategy="Bayesian",
    max_jobs=4, max_parallel_jobs=2, early_stopping_type="Auto",
    base_tuning_job_name="safedrive-fraud-hpo", tags=U.TAGS)
print(list(ranges))

Start the tuning job without blocking. If an earlier run of this notebook already tuned, reuse that job.

In [ ]:
HPO_JOB = U.get_state("fraud_hpo_job")
if not HPO_JOB:
    tuner.fit(inputs, wait=False)
    HPO_JOB = tuner.latest_tuning_job.name
    U.save_state(fraud_hpo_job=HPO_JOB)
print("tuning job:", HPO_JOB)

Wait for it (about 10 minutes for a new job). Watch the counters of completed training jobs.

In [ ]:
def hpo_status():
    t = sm.describe_hyper_parameter_tuning_job(HyperParameterTuningJobName=HPO_JOB)
    print(t["TrainingJobStatusCounters"])
    return t["HyperParameterTuningJobStatus"]

U.wait_until(hpo_status, ok=("Completed",), bad=("Failed", "Stopped"), delay=30, label="tuning job")

> 🖥️ **Console check:** *Amazon SageMaker AI → Training → Hyperparameter tuning jobs → `safedrive-fraud-hpo-...`*.
> Open the **Best training job** tab (winning values and metric) and the **Training jobs** tab (one row per job; a job that AMT stopped early shows as Stopped). The job settings show the strategy and the ranges.

Compare the jobs. Each row is one training job with the values AMT chose and its final AUC.

In [ ]:
from sagemaker.analytics import HyperparameterTuningJobAnalytics
results = HyperparameterTuningJobAnalytics(HPO_JOB, sagemaker_session=sm_session).dataframe()
show = list(ranges) + ["FinalObjectiveValue", "TrainingElapsedTimeSeconds", "TrainingJobStatus"]
results.sort_values("FinalObjectiveValue", ascending=False)[show].round(4)

Get the best training job and its model artifact. This becomes model **v2**, used later for A/B tests
and the model registry.

In [ ]:
best = sm.describe_hyper_parameter_tuning_job(HyperParameterTuningJobName=HPO_JOB)["BestTrainingJob"]
BEST_JOB = best["TrainingJobName"]
model_v2 = sm.describe_training_job(TrainingJobName=BEST_JOB)["ModelArtifacts"]["S3ModelArtifacts"]
print(BEST_JOB, best["FinalHyperParameterTuningJobObjectiveMetric"]["Value"])
print(best["TunedHyperParameters"], model_v2, sep="\n")

**Warm start.** A new tuning job can start from what this one learned instead of from zero.

| Type | Use when |
|---|---|
| `IdenticalDataAndAlgorithm` | Same data and algorithm: you just want more jobs or narrower ranges |
| `TransferLearning` | Data or algorithm version changed, but the old results are still a useful hint |

The cell below only builds the configuration (no cost). You would pass it as `warm_start_config=` to a new tuner.

In [ ]:
from sagemaker.tuner import WarmStartConfig, WarmStartTypes
warm = WarmStartConfig(WarmStartTypes.IDENTICAL_DATA_AND_ALGORITHM, parents={HPO_JOB})
warm.to_input_req()

## 2.2.4 Reduce training time (and cost)

| Technique | How | Note |
|---|---|---|
| **Early stopping** | `early_stopping_rounds` (XGBoost), AMT `early_stopping_type="Auto"` | Stops when the validation metric stops improving |
| **Distributed training** | `instance_count` > 1 | Split the data with `ShardedByS3Key` so each instance gets a different part. Worth it only for large data |
| **Managed Spot** | `use_spot_instances=True`, `max_wait`, `checkpoint_s3_uri` | Up to 90% cheaper; can be interrupted, resumes from the checkpoint |
| **Warm pools** | `keep_alive_period_in_seconds=...` | Keeps instances ready between jobs: no start-up wait (you pay while they are kept) |
| **Input mode** | `FastFile` or `Pipe` instead of `File` | Streams from S3, no long download for big datasets |
| **Bigger or GPU instance** | Instance type, `tree_method="gpu_hist"` | Faster per job, higher price per hour |

A helper that starts a job without waiting and reports a clear message if the account quota is too low
(new accounts sometimes have a quota of 0 for Spot training instances).

In [ ]:
def start(estimator, job_inputs):
    try:
        estimator.fit(job_inputs, wait=False)
        return estimator.latest_training_job.name
    except Exception as e:  # for example ResourceLimitExceeded
        print("could not start:", str(e)[:200])
        return None

print("helper ready")

**📘 Concept: Managed Spot training and checkpoints**

Spot instances are spare AWS capacity at a large discount. AWS can take them back at short notice. With **managed** Spot training, SageMaker requests the capacity, and if the job is interrupted it waits for capacity and starts the job again.

1. `use_spot_instances=True` turns it on.
2. `max_run` = the longest the training itself may run. `max_wait` = the longest you accept in total, **including time waiting for Spot capacity**, so `max_wait` must be ≥ `max_run`.
3. `checkpoint_s3_uri`: SageMaker copies whatever the algorithm writes to the local checkpoint folder (`/opt/ml/checkpoints` by default) to S3, and copies it back when the job restarts. The algorithm or your script must write and reload the checkpoints. Without them, a restarted job begins from zero.
4. You are billed only for the seconds the job actually ran. The job description shows training time and billable time; the difference is your saving.

- Good fit: long, interruption-tolerant jobs and tuning jobs. Poor fit: a job with a hard deadline, because waiting time is not predictable.
- Exam cues: "reduce training cost", "can tolerate interruptions" → managed Spot **with checkpoints**. "Spot job restarts from the beginning" → checkpointing is missing. "Job fails with a wait-time error" → `max_wait` is too small.

**Job B: managed Spot + checkpoints + early stopping.** We allow up to 500 rounds, but training stops
when validation AUC has not improved for 10 rounds. `max_wait` includes the time spent waiting for Spot capacity.

In [ ]:
spot_est = make_xgb("safedrive-fraud-spot", {"num_round": 500, "early_stopping_rounds": 10},
                    use_spot_instances=True, max_run=900, max_wait=1800,
                    checkpoint_s3_uri=U.s3_uri(f"{U.PREFIX}/checkpoints/fraud-spot-{int(time.time())}"))
SPOT_JOB = start(spot_est, inputs)
print("spot job:", SPOT_JOB)

**📘 Concept: Distributed training, data distribution and input modes**

Two separate settings decide how data reaches the training instances.

**Which instance gets which files** (`distribution` on a channel):

| Setting | Meaning | Use for |
|---|---|---|
| `FullyReplicated` (default) | Every instance gets every file | Single instance; or algorithms that split the data themselves |
| `ShardedByS3Key` | The files are divided among the instances | Multi-instance training, so each instance works on its own part. Needs at least as many files as instances |

**How the files arrive** (input mode):

| Mode | How | Choose when |
|---|---|---|
| `File` (default) | Downloads the whole dataset to the instance disk before training starts | Small or medium data; needs enough disk |
| `FastFile` | Files look local but are streamed from S3 when read | Large data, fast start, no code change |
| `Pipe` | Data is streamed through a pipe, read once in order | Large data with algorithms or code written for streaming (for example RecordIO) |

- More instances add start-up and coordination cost. They pay off only when one instance is too slow or the data does not fit; first try a larger instance (or a GPU for deep learning).
- Exam cues: "training waits a long time downloading data" or "dataset larger than the volume" → FastFile or Pipe. "Each instance should train on a different subset" → `ShardedByS3Key`.

**Job C: distributed training on 2 instances.** First split the training file into two parts, so that
`ShardedByS3Key` can give each instance its own part.

In [ ]:
half = len(train) // 2
for i, part in enumerate([train.iloc[:half], train.iloc[half:]]):
    uri = U.write_df(part, f"{C.PROCESSED}/train_shards/part-{i}.csv", header=False)
    print(uri, len(part))

Start the 2-instance job. The validation channel stays `FullyReplicated` (every instance gets all of it).

In [ ]:
dist_inputs = {
    "train": TrainingInput(U.s3_uri(f"{C.PROCESSED}/train_shards/"), content_type="text/csv",
                           distribution="ShardedByS3Key"),
    "validation": inputs["validation"]}
dist_est = make_xgb("safedrive-fraud-dist", instance_count=2)
DIST_JOB = start(dist_est, dist_inputs)
print("distributed job:", DIST_JOB)

Wait for both jobs (about 4 to 6 minutes; a Spot job can wait longer for capacity).

In [ ]:
for job in filter(None, [SPOT_JOB, DIST_JOB]):
    U.wait_until(lambda: sm.describe_training_job(TrainingJobName=job)["TrainingJobStatus"],
                 ok=("Completed",), bad=("Failed", "Stopped"), delay=30, label=job)

> 🖥️ **Console check:** *Amazon SageMaker AI → Training → Training jobs → `safedrive-fraud-spot-...`*.
> In the job summary compare training time with billable time; the console also shows the managed Spot training savings as a percentage. Then open `safedrive-fraud-dist-...` and confirm the instance count of 2.

## 2.1.6 Tradeoff: performance vs training time vs cost

- **Billable seconds × instance count × price per second** = training cost.
- For Spot jobs, billable time is lower than training time: that difference is the saving.
- More instances or a bigger instance cut wall-clock time only when the data is large enough.
  On small data, start-up time dominates and you just pay more.

A function that turns a training job into one summary row. 💰 The price is an approximate on-demand
price for `ml.m5.large` training; check the SageMaker pricing page for your Region.

In [ ]:
PRICE_PER_HOUR = 0.115

def job_row(label, job_name):
    d = sm.describe_training_job(TrainingJobName=job_name)
    n, bill = d["ResourceConfig"]["InstanceCount"], d.get("BillableTimeInSeconds", 0)
    auc = {m["MetricName"]: m["Value"] for m in d.get("FinalMetricDataList", [])}.get("validation:auc")
    return {"config": label, "instances": n, "spot": d.get("EnableManagedSpotTraining", False),
            "train_s": d.get("TrainingTimeInSeconds"), "billable_s": bill,
            "cost_usd": round(bill * n * PRICE_PER_HOUR / 3600, 4), "val_auc": auc}

job_row("A: baseline, 1 instance on-demand", BASE_JOB)

Compare the configurations side by side. Which one would you pick if you retrain every night?

In [ ]:
jobs = [("A: baseline, 1 instance on-demand", BASE_JOB), ("B: Spot + early stopping", SPOT_JOB),
        ("C: 2 instances distributed", DIST_JOB), ("D: best of 4 tuning jobs", BEST_JOB)]
compare = pd.DataFrame([job_row(label, j) for label, j in jobs if j])
compare

**How to read it**
- B usually has a similar AUC for the lowest cost (Spot discount, fewer rounds).
- C costs about twice A with no gain here: 14,000 rows are far too few to need two instances.
- D usually has the best AUC, but remember the tuning job paid for **all four** training jobs, not just the winner.

## 2.2.6 Combine models to improve performance or reduce cost

| Method | Idea | Typical example |
|---|---|---|
| **Bagging** | Many models on random samples, average them (less variance) | Random Forest |
| **Boosting** | Models in sequence, each fixes the errors of the last (less bias) | XGBoost, LightGBM |
| **Voting / averaging** | Average the outputs of different model types | XGBoost + logistic regression |
| **Stacking** | A small "meta" model learns how to mix the base models | Meta logistic regression |
| **Cascade** | Cheap model handles easy cases, expensive model only the uncertain ones | Small model first, large model (or FM) second |

On SageMaker: several containers behind one endpoint = **inference pipeline** (serial) or
**multi-container endpoint**; AutoGluon builds stacked ensembles for you.

Train two different base models locally: a logistic regression and an XGBoost model.

In [ ]:
from sklearn.linear_model import LogisticRegression
logit = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(Xtr, ytr)
booster = xgb.XGBClassifier(n_estimators=150, learning_rate=0.1, max_depth=5).fit(Xtr, ytr)
p_logit, p_xgb = logit.predict_proba(Xva)[:, 1], booster.predict_proba(Xva)[:, 1]
print("logistic AUC", round(roc_auc_score(yva, p_logit), 4), "| xgboost AUC", round(roc_auc_score(yva, p_xgb), 4))

**Averaging and stacking.** Stacking trains a meta-model on out-of-fold predictions of the base models
(`cv=3`), so the meta-model never sees predictions made on a base model's own training rows.

In [ ]:
from sklearn.ensemble import StackingClassifier
stack = StackingClassifier([("logit", logit), ("xgb", booster)], final_estimator=LogisticRegression(),
                           cv=3, stack_method="predict_proba").fit(Xtr, ytr)
preds = {"logistic": p_logit, "xgboost": p_xgb, "average": (p_logit + p_xgb) / 2,
         "stacking": stack.predict_proba(Xva)[:, 1]}
pd.DataFrame({k: {"roc_auc": roc_auc_score(yva, p), "pr_auc": average_precision_score(yva, p)}
              for k, p in preds.items()}).T.round(4)

The synthetic fraud signal is close to linear, so the simple logistic model is already strong here.
That is a lesson too: try the simple, cheap, explainable model first.

**Cascade (cheap model first).** The logistic model scores every claim. Only claims in its "unsure"
band go to the more expensive model. Look at how many calls to the expensive model you save.

In [ ]:
unsure = (p_logit > 0.05) & (p_logit < 0.60)
p_cascade = np.where(unsure, p_xgb, p_logit)
print(f"sent to the expensive model: {unsure.mean():.0%} of claims")
pd.DataFrame({k: {"roc_auc": roc_auc_score(yva, p), "pr_auc": average_precision_score(yva, p)}
              for k, p in {"expensive model on everything": p_xgb, "cascade": p_cascade}.items()}).T.round(4)

The same cascade idea is used with foundation models: a small FM (Nova Micro) answers first, and only
hard or low-confidence requests go to a large FM. Bedrock **Intelligent Prompt Routing** does this for you.

## Exam traps

- Spot training without `checkpoint_s3_uri`: an interruption restarts training from zero. `max_wait` must be ≥ `max_run`.
- Bayesian tuning with all jobs in parallel loses its advantage: it cannot learn from jobs that have not finished.
- Built-in XGBoost CSV input: label in the **first** column, **no header**. A header row silently becomes a data row or fails the job.
- Train score high and validation score low is overfitting: adding model capacity makes it worse.
- More instances do not make small-data training faster; start-up time dominates.

## Quiz

**Q1.** Training accuracy is 99% and validation accuracy is 81%. Which action is most likely to help?
A) Increase `max_depth`  B) Train for more rounds  C) Increase `lambda` and add early stopping  D) Remove the validation set

<details><summary>Answer</summary>C. The model overfits; more regularization and early stopping reduce the gap. A and B make it worse.</details>

**Q2.** A team runs 50-hour training jobs and wants to cut cost by about 70%. Jobs can tolerate interruptions. What should they configure?
A) Managed Spot training with checkpoints to S3  B) A warm pool  C) More parallel tuning jobs  D) Pipe input mode

<details><summary>Answer</summary>A. Spot gives the large discount; checkpoints let an interrupted job resume instead of starting over.</details>

**Q3.** A tuning job with 40 jobs finished last week. The data is unchanged and the team wants to explore
20 more jobs around the best region without starting from scratch. What should they use?
A) Grid search  B) Warm start with `IdenticalDataAndAlgorithm`  C) Warm start with `TransferLearning`  D) Hyperband

<details><summary>Answer</summary>B. Same data and algorithm, so the new job can reuse the old results directly.</details>

**Q4.** A data scientist has a working scikit-learn training script and wants to run it on SageMaker with
the least rework. What is the best option?
A) Rewrite it for a built-in algorithm  B) Build a custom Docker image  C) Script mode with the SageMaker scikit-learn container  D) Use SageMaker Canvas

<details><summary>Answer</summary>C. Script mode runs your script in an AWS-managed framework container; only small changes (arguments, paths) are needed.</details>

**Q5.** The training loss of a neural network swings up and down and sometimes becomes NaN. What should you try first?
A) Lower the learning rate  B) Increase the number of epochs  C) Use a larger instance  D) Remove regularization

<details><summary>Answer</summary>A. Jumping or exploding loss is the sign of a learning rate that is too high.</details>

## Save state
Keys other notebooks use: `fraud_model_data`, `fraud_training_job`, `xgb_image` (saved above) and
`fraud_model_data_v2` (the tuned model).

In [ ]:
state = U.save_state(fraud_model_data_v2=model_v2, fraud_best_training_job=BEST_JOB,
                     fraud_sklearn_model_data=sk.model_data)
{k: state[k] for k in ("fraud_model_data", "fraud_training_job", "fraud_model_data_v2", "fraud_hpo_job")}

## Cleanup
Training and tuning jobs stop by themselves and their instances are gone, so nothing is running.
**Kept for later notebooks:** the model artifacts in `s3://<bucket>/safedrive/models/` (v1 and v2).
The cells below only remove small helper files.

### Before you delete: what later notebooks use

The cleanup cells below **do not delete** the items in this table.

| Kept | Used by later notebooks | If you delete it anyway |
|---|---|---|
| Model artifacts in `safedrive/models/` (v1 and the tuned v2) | 2.3a, 3.1a, 3.2a, 3.3a, 4.1a, and every notebook that uses the fraud endpoint | v1 is retrained by the catch-up cell (about 5 minutes); notebooks that want v2 fall back to v1 |

- **Continuing to the next notebook soon?** Run the cleanup cells and keep the items above. None of them bills while idle (only a little storage).
- **Stopping for good?** Run `99_cleanup/99_teardown.ipynb`; it removes everything, including these.

In [ ]:
import shutil
shutil.rmtree("local_data", ignore_errors=True)
print("removed local_data/")

In [ ]:
s3 = U.client("s3")
for key in U.list_keys(f"{C.PROCESSED}/train_shards/") + U.list_keys(f"{U.PREFIX}/checkpoints/"):
    U.safe(s3.delete_object, Bucket=U.bucket(), Key=key)